In [1]:
from torch.nn.functional import softmax
import torch
import torch.nn as nn
import os
path = os.getcwd()

In [2]:
import tiktoken

In [3]:
state_dict = torch.load(path+'/weights/pytorch_model.bin')

In [4]:
state_dict['wte.weight'].shape

torch.Size([50257, 768])

### Tokenizer

In [5]:
tokenizer = tiktoken.get_encoding('gpt2')

In [6]:
sample_text = "Hello there, how are you?"
tokenizer.encode(sample_text)

[15496, 612, 11, 703, 389, 345, 30]

In [7]:
 torch.tensor(tokenizer.encode('Hello, you!'))

tensor([15496,    11,   345,     0])

### Load token embedding weights into nn.Embedding layer

In [8]:
wte = nn.Embedding(50257, 768)
wte.weight

Parameter containing:
tensor([[-1.1468, -1.1954, -0.8180,  ...,  0.2753,  1.1836,  0.1613],
        [-0.4747,  0.1234,  0.6917,  ...,  0.8933,  0.7405, -0.0791],
        [ 1.0570,  2.0459, -0.4412,  ...,  0.5426, -0.1975,  0.1562],
        ...,
        [ 0.5014, -0.8879, -0.9011,  ...,  0.2022, -0.6304, -0.2422],
        [ 1.6671,  0.1822, -2.5040,  ...,  0.2312, -0.8429,  0.2168],
        [ 0.0514,  0.8155,  0.4687,  ...,  1.1874, -0.5892, -1.7054]],
       requires_grad=True)

In [9]:
wte.load_state_dict({'weight': state_dict['wte.weight']})

<All keys matched successfully>

### Check that it worked

In [10]:
wte.weight

Parameter containing:
tensor([[-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453],
        [ 0.0403, -0.0486,  0.0462,  ...,  0.0861,  0.0025,  0.0432],
        [-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
        ...,
        [-0.0445, -0.0548,  0.0123,  ...,  0.1044,  0.0978, -0.0695],
        [ 0.1860,  0.0167,  0.0461,  ..., -0.0963,  0.0785, -0.0225],
        [ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207]],
       requires_grad=True)

In [11]:
state_dict['wte.weight']

tensor([[-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453],
        [ 0.0403, -0.0486,  0.0462,  ...,  0.0861,  0.0025,  0.0432],
        [-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
        ...,
        [-0.0445, -0.0548,  0.0123,  ...,  0.1044,  0.0978, -0.0695],
        [ 0.1860,  0.0167,  0.0461,  ..., -0.0963,  0.0785, -0.0225],
        [ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207]])

In [12]:
tokens = torch.tensor(tokenizer.encode('Hello, you!'))
tokens

tensor([15496,    11,   345,     0])

In [13]:
wte(tokens)

tensor([[-0.0687, -0.1327,  0.0112,  ...,  0.0715, -0.0297, -0.0477],
        [ 0.0115, -0.0029,  0.0323,  ...,  0.0277, -0.0297, -0.0599],
        [-0.0337,  0.0484,  0.0309,  ..., -0.1242, -0.0810, -0.0539],
        [-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453]],
       grad_fn=<EmbeddingBackward0>)

In [14]:
wte(tokens).shape

torch.Size([4, 768])

In [15]:
Q_proj = nn.Linear(in_dim, out_dim, bias=False)
K_proj = nn.Linear(in_dim, out_dim, bias=False)
V_proj = nn.Linear(in_dim, out_dim, bias=False)

NameError: name 'in_dim' is not defined

So we need `n_tok`x`n_embed` QKV matrices. That's *after* the projections.
The embedded tokens (i.e. wte(tokens)) has that shape already. What do we feed into the QKV projections?
Ah, just feed those. They're sqaure matrices so dimensions are preserved.

### Load positional embeddings into wpe nn.Embedding layer

In [27]:
wpe = nn.Embedding(1024, 768)
wpe.load_state_dict({'weight': state_dict['wpe.weight']})

<All keys matched successfully>

In [32]:
wte(tokens) + wpe(torch.arange(4))

torch.Size([4, 768])

In [38]:
dropout = nn.Dropout(0.1)

### Partial forward pass of TransformerDecoder

In [39]:
embeddings = wte(tokens) + wpe(torch.arange(4))
x = dropout(embeddings)

In [42]:
x.shape

torch.Size([4, 768])

That's ready to go into the blocks.

### Load the params for the Transformer blocks

First, layernorm.

In [47]:
ln_1 = nn.LayerNorm(768, eps=1e-05)
ln_1.load_state_dict({
    'weight': state_dict['h.0.ln_1.weight'],
    'bias': state_dict['h.0.ln_1.bias']
})

<All keys matched successfully>

### Check that it's the same

In [48]:
ln_1.weight

Parameter containing:
tensor([0.2232, 0.1820, 0.1534, 0.1917, 0.2036, 0.1948, 0.1467, 0.1865, 0.2143,
        0.1956, 0.2118, 0.2153, 0.1882, 0.2074, 0.1871, 0.2040, 0.2044, 0.1900,
        0.1952, 0.0475, 0.1909, 0.2115, 0.1971, 0.2202, 0.1998, 0.2108, 0.2303,
        0.1879, 0.1939, 0.2018, 0.1891, 0.1861, 0.1958, 0.1832, 0.1978, 0.2243,
        0.0706, 0.1958, 0.1943, 0.1939, 0.1978, 0.1951, 0.1995, 0.1912, 0.2083,
        0.2037, 0.1849, 0.1945, 0.2189, 0.0419, 0.1977, 0.1979, 0.0608, 0.1824,
        0.2055, 0.0476, 0.1892, 0.2079, 0.2047, 0.2233, 0.2097, 0.2075, 0.2076,
        0.1793, 0.1312, 0.1841, 0.1939, 0.1561, 0.0577, 0.1948, 0.2048, 0.1717,
        0.1942, 0.1708, 0.1989, 0.1993, 0.2082, 0.1071, 0.1968, 0.1770, 0.2164,
        0.1864, 0.1938, 0.2184, 0.1343, 0.1707, 0.0683, 0.1401, 0.1823, 0.2045,
        0.2007, 0.1853, 0.1783, 0.1889, 0.1870, 0.1975, 0.2114, 0.2108, 0.2083,
        0.2409, 0.1938, 0.2022, 0.0857, 0.1823, 0.1879, 0.1979, 0.1850, 0.1029,
        0.1762, 0.

In [49]:
state_dict['h.0.ln_1.weight']

tensor([0.2232, 0.1820, 0.1534, 0.1917, 0.2036, 0.1948, 0.1467, 0.1865, 0.2143,
        0.1956, 0.2118, 0.2153, 0.1882, 0.2074, 0.1871, 0.2040, 0.2044, 0.1900,
        0.1952, 0.0475, 0.1909, 0.2115, 0.1971, 0.2202, 0.1998, 0.2108, 0.2303,
        0.1879, 0.1939, 0.2018, 0.1891, 0.1861, 0.1958, 0.1832, 0.1978, 0.2243,
        0.0706, 0.1958, 0.1943, 0.1939, 0.1978, 0.1951, 0.1995, 0.1912, 0.2083,
        0.2037, 0.1849, 0.1945, 0.2189, 0.0419, 0.1977, 0.1979, 0.0608, 0.1824,
        0.2055, 0.0476, 0.1892, 0.2079, 0.2047, 0.2233, 0.2097, 0.2075, 0.2076,
        0.1793, 0.1312, 0.1841, 0.1939, 0.1561, 0.0577, 0.1948, 0.2048, 0.1717,
        0.1942, 0.1708, 0.1989, 0.1993, 0.2082, 0.1071, 0.1968, 0.1770, 0.2164,
        0.1864, 0.1938, 0.2184, 0.1343, 0.1707, 0.0683, 0.1401, 0.1823, 0.2045,
        0.2007, 0.1853, 0.1783, 0.1889, 0.1870, 0.1975, 0.2114, 0.2108, 0.2083,
        0.2409, 0.1938, 0.2022, 0.0857, 0.1823, 0.1879, 0.1979, 0.1850, 0.1029,
        0.1762, 0.1953, 0.2231, 0.2006, 

Run input through it:

In [54]:
x = ln_1(x)

Now we can do MHA

In [16]:
def scaled_dot_product_attention(queries, keys, values, mask=None):
    """
    - Q, K, V each (n_tokens x out_dim)
    - Q @ K.T => (n_tokens x n_tokens) (similarity score)
    - (Q @ K.T) @ V => (QK^T: n_tokens x n_tokens) x (V: n_tokens x out_dim)
                    => (n_tokens x out_dim) returned (attn)
    - DIM NOT CHANGED by SDP
    """
    similarity_score = queries.matmul(keys.T)
    dk = keys.size(-1)
    denom = torch.sqrt(torch.tensor(dk))
    sdp = 1/denom * similarity_score
    
    if mask is not None:
        n_tokens = keys.size(-2) # <= n_ctx
        mask_val = torch.finfo(sdp.dtype).min # something like -3.4e38 for float32
        masked_attn_weights = torch.where(mask[:n_tokens, :n_tokens], sdp, mask_val)
        sdp = masked_attn_weights

    attn = torch.matmul(softmax(sdp, dim=-1), values)
    return attn

class AttentionHead(nn.Module):
    def __init__(self, in_dim, out_dim): # both equal to embedding_dim- no, out_dim=head_dim
        super().__init__()
        """
        - DIM NOT CHANGED by these
        - in: n_tokens x embed_dim
        - out: n_tokens x embed_dim
        TODO: batch index
        """
        self.Q_proj = nn.Linear(in_dim, out_dim)#, bias=False)
        self.K_proj = nn.Linear(in_dim, out_dim)#, bias=False)
        self.V_proj = nn.Linear(in_dim, out_dim)#, bias=False)
        
    def forward(self, x, mask=None): # n_tokens x embed_dim
        Q = self.Q_proj(x) # (x: n_tokens x embed_dim) @ (Q_proj: embed_dim x embed_dim)^T -> (n_tokens x embed_dim)
        K = self.K_proj(x) # dim same as x (")
        V = self.V_proj(x) # dim same as x (")
        return scaled_dot_product_attention(Q, K, V, mask=mask)

In [156]:
attn_head_0 = AttentionHead(in_dim=768, out_dim=768)
attn_head_0.state_dict()

OrderedDict([('Q_proj.weight',
              tensor([[ 1.3800e-02, -1.8386e-02,  2.7525e-02,  ..., -3.6162e-03,
                       -2.8565e-02, -1.8675e-02],
                      [ 3.3628e-02, -1.1045e-03,  5.7228e-05,  ..., -2.5072e-02,
                       -1.8025e-02, -3.4801e-02],
                      [-4.8107e-04,  1.8098e-02, -1.4548e-03,  ..., -3.2055e-02,
                       -2.1684e-02, -2.0017e-02],
                      ...,
                      [-5.8224e-03,  3.0439e-02,  4.7041e-03,  ...,  2.6257e-03,
                        3.0607e-02, -2.5751e-02],
                      [-2.5575e-02,  1.8881e-02,  3.4557e-02,  ..., -2.7341e-02,
                        3.1643e-02,  1.1224e-02],
                      [-1.1904e-02, -3.4392e-03,  2.9532e-02,  ..., -3.0940e-02,
                       -3.3397e-02,  8.9756e-03]])),
             ('Q_proj.bias',
              tensor([ 1.3533e-03, -6.5840e-03, -2.4449e-02, -2.8397e-03,  3.1392e-02,
                       2.2800e-02, -2

In [157]:
state_dict['h.0.attn.c_attn.bias']

tensor([ 0.4803, -0.5254, -0.4293,  ...,  0.0126, -0.0499,  0.0032])

In [158]:
attn_head_0.Q_proj.bias.shape

torch.Size([768])

h.0.attn.c_attn.weight ==> [768, 2304]


h.0.attn.c_attn.bias ==> [2304]

In [159]:
2304/768

3.0

3 because it's housing Q, K and V projections.
https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt2/modeling_gpt2.py#L293C1-L294C1

In [160]:
Q_proj_w, K_proj_w, V_proj_w = state_dict['h.0.attn.c_attn.weight'].split(768, dim=1)
# split weights along dim=1 which is the 2304 (see above)
Q_proj_b, K_proj_b, V_proj_b = state_dict['h.0.attn.c_attn.bias'].split(768, dim=0)
# split biases along dim=0 which is also 2304 (see above)

In [161]:
Q_proj_w.shape, K_proj_b.shape

(torch.Size([768, 768]), torch.Size([768]))

### Test loading QKV weights into dummy head

In [162]:
attn_head_0.load_state_dict({
    'Q_proj.weight': Q_proj_w,
    'K_proj.weight': K_proj_w,
    'V_proj.weight': V_proj_w,
    'Q_proj.bias': Q_proj_b,
    'K_proj.bias': K_proj_b,
    'V_proj.bias': V_proj_b
})

<All keys matched successfully>

In [163]:
attn_head_0(x).shape

torch.Size([4, 768])

But now I'm confused. After splitting, we get the three matrices, but what about MHA?
Shouldn't there be 12 of these heads per block?

### Realize we have to split the QKV matrices further from 768 to 64 times 12 

In [164]:
768/12

64.0

That's it; the heads are 64x64 not 768x768. So there *are* 12 of them.

In [17]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, in_dim, out_dim, n_ctx):
        super().__init__()
        self.head_dim = out_dim // n_heads # 768/12 = 64 head_dim
        self.attn_heads = nn.ModuleList([
            # each head takes in (n_tokens x head_dim)
            # and returns        (n_tokens x head_dim)
            AttentionHead(in_dim, self.head_dim) for _ in range(n_heads)
        ])
        self.register_buffer(
            "mask",
            torch.tril(torch.ones((n_ctx, n_ctx), dtype=torch.bool)),
            persistent=False
        )

        self.out_proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        n_tokens, in_dim = x.shape
        heads = torch.concat([head(x, mask=self.mask) for head in self.attn_heads], dim=-1)
        # = n_tokens x (head_dim*n_heads = out_dim = embed_dim)
        out = self.out_proj(heads)
        return out

In [166]:
mha = MultiHeadAttention(n_heads=12, in_dim=768, out_dim=768, n_ctx=1024)

In [167]:
mha.load_state_dict({
    'out_proj.weight': state_dict['h.0.attn.c_proj.weight'],
    'out_proj.bias': state_dict['h.0.attn.c_proj.bias'],
}, strict=False)

# split from 2304 to three with dim 768 (64 * 12 heads)
Q_proj_w, K_proj_w, V_proj_w = state_dict['h.0.attn.c_attn.weight'].split(768, dim=1)
Q_proj_b, K_proj_b, V_proj_b = state_dict['h.0.attn.c_attn.bias'].split(768, dim=0)

# for each head, split its weights from the full matrix, then load into MHA state_dict
for i in range(12):
    mha.load_state_dict({
        f'attn_heads.{i}.Q_proj.weight': Q_proj_w.split(64, dim=1)[i].T,
        f'attn_heads.{i}.K_proj.weight': K_proj_w.split(64, dim=1)[i].T,
        f'attn_heads.{i}.V_proj.weight': V_proj_w.split(64, dim=1)[i].T,
        f'attn_heads.{i}.Q_proj.bias': Q_proj_b.split(64, dim=0)[i],
        f'attn_heads.{i}.K_proj.bias': K_proj_b.split(64, dim=0)[i],
        f'attn_heads.{i}.V_proj.bias': V_proj_b.split(64, dim=0)[i]
    }, strict=False)

In [168]:
mha.attn_heads[0].Q_proj.weight

Parameter containing:
tensor([[-0.4738,  0.0874,  0.0039,  ..., -0.2592,  0.1517, -0.4100],
        [-0.2614,  0.1473,  0.0695,  ..., -0.0164,  0.2170, -0.1924],
        [-0.0978,  0.2387,  0.3668,  ...,  0.1991,  0.1043, -0.2400],
        ...,
        [ 0.0908, -0.3679, -0.2476,  ...,  0.0801, -0.0959, -0.3557],
        [ 0.2785,  0.3194,  0.1122,  ..., -0.0739, -0.5090, -0.1824],
        [ 0.2262, -0.0895,  0.2564,  ...,  0.0586, -0.2666, -0.2051]],
       requires_grad=True)

In [169]:
state_dict['h.0.attn.c_attn.weight'].T

tensor([[-0.4738,  0.0874,  0.0039,  ..., -0.2592,  0.1517, -0.4100],
        [-0.2614,  0.1473,  0.0695,  ..., -0.0164,  0.2170, -0.1924],
        [-0.0978,  0.2387,  0.3668,  ...,  0.1991,  0.1043, -0.2400],
        ...,
        [ 0.0513, -0.0525,  0.1143,  ...,  0.0095,  0.0293, -0.0046],
        [-0.0584, -0.0113,  0.0363,  ..., -0.0516, -0.0429,  0.0070],
        [ 0.0250, -0.0156, -0.0318,  ...,  0.0319, -0.0475,  0.0198]])

In [171]:
mha(x) # this is only the MHA of *one* block

tensor([[-0.8509,  0.0349,  0.0574,  ...,  0.0599,  0.2817, -1.5635],
        [-0.6682, -0.1846, -0.5916,  ..., -0.5374,  0.2624, -1.1613],
        [-0.8116, -0.1709, -0.7335,  ..., -0.1890,  0.2931, -1.5012],
        [ 0.0148, -0.1071, -0.6473,  ..., -0.7319, -0.0674, -1.0896]],
       grad_fn=<AddmmBackward0>)

Now we need the rest of the forward pass:
- dropout again (wait, was there dropout before MHA? there isn't in the old implementation
  - ah yes, in the Transformer it was, before the block which houses the attention
- residual (add input + positional embeddings again)
- layernorm2
- MLP
- dropout again
- residual again (connection after first dropout in block
- then do this for all 12 blocks
- then a linear layer from `n_embed` to `vocab_size` (no bias(?))
- then softmax and compare logits to hugginface sanity check

In [18]:
GPT2Config = {
  "activation_function": "gelu_new", # 'new'?
  "architectures": [
    "GPT2LMHeadModel" # anything special bout this?
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256, # same as eos?
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_layer": 12,
  "n_positions": 1024,
  "resid_pdrop": 0.1,
  "summary_activation": None,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": True,
  "summary_type": "cls_index",
  "summary_use_proj": True,
  "task_specific_params": {
    "text-generation": {
      "do_sample": True,
      "max_length": 50
    }
  },
  "vocab_size": 50257
}

In [19]:
class TransformerBlock(nn.Module):
    
    def __init__(self, n_heads, in_dim, out_dim, n_ctx):
        super().__init__()
        self.mha = MultiHeadAttention(n_heads, in_dim, out_dim, n_ctx) # returns in_dim x dk*n_heads
        self.layernorm1 = nn.LayerNorm(GPT2Config['n_embd'], eps=GPT2Config['layer_norm_epsilon'])
        self.layernorm2 = nn.LayerNorm(GPT2Config['n_embd'], eps=GPT2Config['layer_norm_epsilon'])
        self.dropout = nn.Dropout(GPT2Config['attn_pdrop'])
        
        self.ffn = nn.Sequential(
            nn.Linear(in_dim, in_dim * 4),
            nn.GELU(approximate='tanh'),
            nn.Linear(in_dim * 4, in_dim)
        )

    def forward(self, x):
        x_old = x
        x = self.layernorm1(x)
        x = self.mha(x)
        x = self.dropout(x)
        x = x + x_old
        
        x_old_2 = x
        x = self.layernorm2(x)
        x = self.ffn(x)
        x = self.dropout(x)
        x = x + x_old_2
        return x

In [20]:
class TransformerDecoder(nn.Module):
    
    def __init__(self, n_layers, n_heads, vocab_size, n_embed, n_ctx):
        super().__init__()
        self.ctx_len = n_ctx
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.positional_embedding = nn.Embedding(n_ctx, n_embed)
        self.dropout = nn.Dropout(0.1)
        
        self.blocks = nn.Sequential(*[
            TransformerBlock(n_heads, n_embed, n_embed, n_ctx) 
            for _ in range(n_layers)
        ])
        self.layernorm_final = nn.LayerNorm(GPT2Config['n_embd'], eps=GPT2Config['layer_norm_epsilon'])
        self.linear = nn.Linear(n_embed, vocab_size, bias=False) # transpose of token_embedding; weights tied
        
        self.softmax = nn.Softmax(dim=0)
        
    def forward(self, x):
        seq_len = x.shape[-1]
        assert seq_len <= self.ctx_len
        positions = torch.arange(seq_len) # 0, 1, 2, .., seq_len-1
        
        embeddings = self.token_embedding(x) + self.positional_embedding(positions)
        
        x = self.dropout(embeddings)
        x = self.blocks(x)
        x = self.layernorm_final(x)
        x = self.linear(x)
        
        return self.softmax(x)

In [21]:
transformer = TransformerDecoder(
    n_layers=12, 
    n_heads=12, 
    vocab_size=50257, 
    n_embed=768, 
    n_ctx=1024
)
transformer.state_dict()['blocks.0.mha.attn_heads.0.Q_proj.weight']

tensor([[-0.0294,  0.0196, -0.0126,  ..., -0.0003, -0.0134, -0.0336],
        [-0.0260,  0.0127,  0.0304,  ...,  0.0099,  0.0042, -0.0348],
        [ 0.0002,  0.0214,  0.0162,  ...,  0.0211, -0.0169,  0.0052],
        ...,
        [-0.0157,  0.0083,  0.0077,  ..., -0.0133,  0.0190,  0.0055],
        [-0.0039, -0.0026,  0.0005,  ...,  0.0187, -0.0214, -0.0011],
        [-0.0172, -0.0269,  0.0332,  ..., -0.0013,  0.0017, -0.0353]])

In [22]:
transformer.state_dict()['blocks.0.layernorm1.weight'].shape[-1] == 768

True

In [23]:
transformer.token_embedding.weight

Parameter containing:
tensor([[ 1.5433, -1.1398,  0.3359,  ...,  1.0434, -0.7420,  0.4966],
        [ 1.2262, -0.1663,  0.6347,  ..., -0.2018,  0.5579, -1.1466],
        [-0.1543, -1.7073, -1.5624,  ...,  0.0673,  0.1420, -1.6899],
        ...,
        [-0.1068, -0.6242,  0.3336,  ...,  1.3663, -2.3448,  0.1285],
        [-1.3536, -0.2041,  0.8033,  ..., -0.4680, -1.1167,  0.3094],
        [-0.5410,  0.1133,  0.9409,  ..., -0.0503, -0.6350, -1.0280]],
       requires_grad=True)

In [24]:
from typing import OrderedDict

def _load_weights(transformer: TransformerDecoder, 
                  state_dict: OrderedDict,
                  n_layers: int,
                  n_heads: int):
    # load embedding matrices
    transformer.load_state_dict({
        f'token_embedding.weight': state_dict[f'wte.weight'],
        f'positional_embedding.weight': state_dict[f'wpe.weight'],
        f'linear.weight': state_dict[f'wte.weight'], # tied/shared with input embedding weights; see markdown below
    }, strict=False)

    # load last layernorm
    transformer.load_state_dict({
        f'layernorm_final.weight': state_dict[f'ln_f.weight'],
        f'layernorm_final.bias': state_dict[f'ln_f.bias'],
    }, strict=False)

    # for each block,
    # load the two layernorms, 
    # the projection weights 
    # and the attention QKV params for each head
    # (skip the mask in state_dict['h.0.attn.bias'] since it's the same..)
    
    for block_idx in range(n_layers):
        transformer.load_state_dict({
            f'blocks.{block_idx}.layernorm1.weight': state_dict[f'h.{block_idx}.ln_1.weight'],
            f'blocks.{block_idx}.layernorm1.bias': state_dict[f'h.{block_idx}.ln_1.bias'],
            f'blocks.{block_idx}.layernorm2.weight': state_dict[f'h.{block_idx}.ln_2.weight'],
            f'blocks.{block_idx}.layernorm2.bias': state_dict[f'h.{block_idx}.ln_2.bias'],
            f'blocks.{block_idx}.mha.out_proj.weight': state_dict[f'h.{block_idx}.attn.c_proj.weight'],
            f'blocks.{block_idx}.mha.out_proj.bias': state_dict[f'h.{block_idx}.attn.c_proj.bias'],
        }, strict=False)
            
        # split from 2304 to three with dim 768 (64 * 12 heads in each matrix)
        Q_proj_w, K_proj_w, V_proj_w = state_dict[f'h.{block_idx}.attn.c_attn.weight'].split(768, dim=1)
        Q_proj_b, K_proj_b, V_proj_b = state_dict[f'h.{block_idx}.attn.c_attn.bias'].split(768, dim=0)
        
        # for each head, split its weights from the full matrix, then load into MHA state_dict
        for head_idx in range(n_heads):
            transformer.load_state_dict({
                f'blocks.{block_idx}.mha.attn_heads.{head_idx}.Q_proj.weight': Q_proj_w.split(64, dim=1)[head_idx].T,
                f'blocks.{block_idx}.mha.attn_heads.{head_idx}.K_proj.weight': K_proj_w.split(64, dim=1)[head_idx].T,
                f'blocks.{block_idx}.mha.attn_heads.{head_idx}.V_proj.weight': V_proj_w.split(64, dim=1)[head_idx].T,
                f'blocks.{block_idx}.mha.attn_heads.{head_idx}.Q_proj.bias': Q_proj_b.split(64, dim=0)[head_idx],
                f'blocks.{block_idx}.mha.attn_heads.{head_idx}.K_proj.bias': K_proj_b.split(64, dim=0)[head_idx],
                f'blocks.{block_idx}.mha.attn_heads.{head_idx}.V_proj.bias': V_proj_b.split(64, dim=0)[head_idx]
            }, strict=False)

In [25]:
transformer = TransformerDecoder(
    n_layers=12, 
    n_heads=12, 
    vocab_size=50257, 
    n_embed=768, 
    n_ctx=1024
)
transformer.state_dict()['blocks.0.mha.attn_heads.0.Q_proj.weight']

tensor([[-0.0079, -0.0341, -0.0207,  ..., -0.0286,  0.0272,  0.0065],
        [-0.0056, -0.0275, -0.0217,  ...,  0.0044, -0.0339, -0.0178],
        [-0.0318,  0.0024,  0.0004,  ...,  0.0217, -0.0301, -0.0153],
        ...,
        [ 0.0203, -0.0153,  0.0076,  ..., -0.0187, -0.0049, -0.0004],
        [-0.0121, -0.0077, -0.0304,  ...,  0.0285,  0.0130,  0.0054],
        [ 0.0283,  0.0071, -0.0160,  ..., -0.0059,  0.0049, -0.0123]])

In [26]:
_load_weights(transformer, state_dict, n_layers=12, n_heads=12)

In [27]:
transformer.state_dict()['blocks.0.mha.attn_heads.0.Q_proj.weight']

tensor([[-0.4738,  0.0874,  0.0039,  ..., -0.2592,  0.1517, -0.4100],
        [-0.2614,  0.1473,  0.0695,  ..., -0.0164,  0.2170, -0.1924],
        [-0.0978,  0.2387,  0.3668,  ...,  0.1991,  0.1043, -0.2400],
        ...,
        [ 0.0908, -0.3679, -0.2476,  ...,  0.0801, -0.0959, -0.3557],
        [ 0.2785,  0.3194,  0.1122,  ..., -0.0739, -0.5090, -0.1824],
        [ 0.2262, -0.0895,  0.2564,  ...,  0.0586, -0.2666, -0.2051]])

In [28]:
state_dict['h.0.attn.c_attn.weight'].split(768, dim=1)[0].split(64, dim=1)[0].T

tensor([[-0.4738,  0.0874,  0.0039,  ..., -0.2592,  0.1517, -0.4100],
        [-0.2614,  0.1473,  0.0695,  ..., -0.0164,  0.2170, -0.1924],
        [-0.0978,  0.2387,  0.3668,  ...,  0.1991,  0.1043, -0.2400],
        ...,
        [ 0.0908, -0.3679, -0.2476,  ...,  0.0801, -0.0959, -0.3557],
        [ 0.2785,  0.3194,  0.1122,  ..., -0.0739, -0.5090, -0.1824],
        [ 0.2262, -0.0895,  0.2564,  ...,  0.0586, -0.2666, -0.2051]])

Done, works!

In [308]:
for key in state_dict.keys():
    if key[0] != 'h':
        print(key)

wte.weight
wpe.weight
ln_f.weight
ln_f.bias


Turns out we're missing the final linear layer after the blocks.
The "LM head" specific to language generation (as opposed to sentence classification or summary).

This means the model is the base GPT-2, not the language generation one.

Nevermind. The final linear layer/LM head is where that weight-tying comes in. `token_embedding.weight` has dimension `vocab_size`x`n_embed` and the LM head has dimensions of `n_embed`x`vocab_size` (sort of just "unembedding"). So it's just those same weights as the input embedding but *transposed*.

In [309]:
transformer.linear.weight.shape

torch.Size([50257, 768])

In [295]:
state_dict['wte.weight'].shape

torch.Size([50257, 768])

In [301]:
transformer.blocks[0].ffn[0]

Linear(in_features=768, out_features=3072, bias=True)

In [302]:
transformer.blocks[0].ffn[0].weight.shape

torch.Size([3072, 768])

The weight matrix is already transposed automatically by pytorch's nn.Linear, so we don't need to transpose the nn.Embedding weight for the nn.Linear if we want to share it.

In [312]:
transformer.linear

Linear(in_features=768, out_features=50257, bias=False)

In [313]:
transformer.linear.weight.shape

torch.Size([50257, 768])

DONE!

In [39]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
        
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[-context_size:]
        
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)
        
        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[-1, :]

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)
        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=0)  # (batch, n_tokens+1)
    return idx

In [40]:
token_input = torch.tensor(tokenizer.encode("Hello, I"))
transformer.eval()
res = generate_text_simple(transformer, token_input, max_new_tokens=6, context_size=1024)
print(res)
print("Generated text:\n\n", tokenizer.decode(res.tolist()))

tensor([0.1361, 0.2578, 0.1267,  ..., 0.4238, 0.0441, 0.1805])
tensor([15496,    11,   314, 13808, 13808, 49510, 49510, 44683, 44683])
Generated text:

 Hello, IissaissaitriitriProofProof


In [36]:
token_input

tensor([15496,    11,   314])